# Unit 4 Assignment: Evaluated Agentic RAG System

**Name:** Anushka Mandal

**SRN:** PES2UG23CS083

**Date:** 24 April 2026


## Overview
This notebook implements a self-evaluating agentic RAG system using:
- **LangChain + FAISS** for the vector store / retrieval
- **CrewAI** for the multi-agent pipeline
- **DeepEval** for automated answer quality evaluation
- **Groq (LLaMA-3.3-70b)** as the LLM backbone

### System Architecture
```
USER QUESTION
      │
      ▼
Agent 1: RAG Retriever  →  (answer + context)
      │
      ▼
Agent 2: Quality Evaluator  →  PASS / FAIL
      │
   ┌──┴──┐
 PASS   FAIL
   │      │
   ▼      ▼
Final  Agent 3: Revisor  →  Final Answer (revised)
```

## Setup: Install Dependencies

In [1]:
# Install all required packages
!pip install -q crewai crewai-tools
!pip install -q langchain langchain-community langchain-groq
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q deepeval
!pip install -q groq
!pip install -q python-dotenv

print("✅ All packages installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.1/811.1 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -q litellm
print("✅ LiteLLM installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 80.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 1.1.1 requires posthog<6.0.0,>=2.4.0, but you have posthog 7.14.0 which is incompatible.
crewai 1.14.4 requires openai<3,>=2.30.0, but you have openai 2.24.0 which is incompatible.
ope

In [3]:
from tenacity import retry, wait_fixed, stop_after_attempt, retry_if_exception_type
import litellm

# Auto-retry on rate limit errors
litellm.num_retries = 5
litellm.retry_after = 10  # wait 10s between retries

## Configure API Keys

In [4]:
import os
from google.colab import userdata

# ─── Option A: Use Colab Secrets (recommended) ───────────────────────────────
# Add GROQ_API_KEY in Colab → Secrets (🔑 icon on left sidebar)
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ Loaded GROQ_API_KEY from Colab Secrets")
except Exception:
    # ─── Option B: Paste directly ────────────────────────────────────────────
    os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"  # <-- replace if needed
    print("⚠️  Using hardcoded API key — consider using Colab Secrets instead")

# DeepEval uses the same Groq key via a custom LLM (configured below)
# No separate DeepEval account needed for this assignment
print(f"GROQ key set: {'✅' if os.environ.get('GROQ_API_KEY') else '❌'}")

✅ Loaded GROQ_API_KEY from Colab Secrets
GROQ key set: ✅


---
# Part 1: Knowledge Base

**Topic chosen: The James Webb Space Telescope (JWST)**

I chose JWST because:
1. It has rich, factual content well-suited for RAG (dates, specs, discoveries)
2. Facts are verifiable and distinct — good for testing faithfulness
3. Adversarial questions (e.g., about Hubble) are clearly out-of-scope

The knowledge base contains ~800 words covering JWST's design, launch, instruments, and key discoveries.

In [5]:
# ─── Knowledge Base Text ─────────────────────────────────────────────────────
KNOWLEDGE_BASE_TEXT = """
The James Webb Space Telescope (JWST) is a space telescope designed to conduct infrared
astronomy. As the largest optical telescope in space, its high infrared resolution and
sensitivity allow it to view objects too old, distant, or faint for the Hubble Space Telescope.

JWST was launched on December 25, 2021, on an Ariane 5 rocket from the Guiana Space Centre
in Kourou, French Guiana. The telescope reached its destination, the Sun-Earth L2 Lagrange
point, approximately 1.5 million kilometers from Earth, in January 2022.

The telescope has a primary mirror consisting of 18 hexagonal gold-plated beryllium segments
with a combined diameter of 6.5 meters. This is significantly larger than Hubble's 2.4-meter
mirror. The mirror segments are made of beryllium because it is lightweight, strong, and
maintains its shape at extremely cold temperatures.

JWST is equipped with four main scientific instruments:
1. NIRCam (Near Infrared Camera): The primary imager, covering wavelengths from 0.6 to 5 micrometers.
2. NIRSpec (Near Infrared Spectrograph): Can observe 100 objects simultaneously, covering 0.6 to 5.3 micrometers.
3. MIRI (Mid-Infrared Instrument): Covers mid-infrared wavelengths from 5 to 28 micrometers.
4. FGS/NIRISS (Fine Guidance Sensor / Near InfraRed Imager and Slitless Spectrograph): Used for precise pointing and exoplanet studies.

JWST operates at extremely cold temperatures, around minus 233 degrees Celsius (40 Kelvin),
to detect faint infrared signals. A five-layer sunshield the size of a tennis court (21 by 14
meters) protects the telescope from the heat of the Sun, Earth, and Moon.

The telescope was developed through a collaboration between NASA, the European Space Agency
(ESA), and the Canadian Space Agency (CSA). The project cost approximately 10 billion USD,
making it one of the most expensive scientific instruments ever built.

Development of JWST began in 1996, with the original launch date planned for 2007. The
project experienced numerous delays and cost overruns over the years. The telescope was
named after James E. Webb, who led NASA from 1961 to 1968 during the critical Mercury and
Gemini programs and the beginning of the Apollo program.

In July 2022, NASA released the first full-color images and spectroscopic data from JWST.
These included an image of the galaxy cluster SMACS 0723, which showed gravitational lensing
of extremely distant background galaxies. Another early image showed the Carina Nebula,
revealing previously invisible details of star-forming regions.

One of JWST's major scientific goals is to observe the formation of the first stars and
galaxies after the Big Bang, during the period known as Cosmic Dawn. By observing in infrared,
JWST can see through dust clouds that would block visible light, and can detect the redshifted
light from the earliest cosmic structures.

JWST has made groundbreaking discoveries about exoplanet atmospheres. In 2022, it detected
carbon dioxide in the atmosphere of WASP-39b, an exoplanet about 700 light-years away. This
was the first clear detection of carbon dioxide in an exoplanet atmosphere, demonstrating
JWST's ability to study planetary atmospheres in detail.

The telescope also studied Stephan's Quintet, a group of five galaxies (four of which are
in a true cluster), capturing unprecedented detail about galactic interactions. JWST revealed
shock waves and gas flows created as one galaxy crashes through the group.

JWST has a design lifetime of 10 years, but the precision of the Ariane 5 launch means it
has enough fuel for potentially more than 20 years of operations. This extended lifetime
gives astronomers much more time to use the telescope than originally planned.

The telescope's data is processed at the Space Telescope Science Institute (STScI) in
Baltimore, Maryland. After a proprietary period, all JWST data is made publicly available
to researchers worldwide.

JWST complements rather than replaces the Hubble Space Telescope. While Hubble primarily
observes in ultraviolet and visible light, JWST specializes in infrared. Together, they
provide coverage across a much broader range of the electromagnetic spectrum.

Early JWST results have already challenged some existing cosmological models. The telescope
found galaxies in the early universe that appear more massive and mature than expected based
on current understanding of galaxy formation. These findings may require revisions to
standard cosmological models.
"""

print(f"Knowledge base loaded: {len(KNOWLEDGE_BASE_TEXT.split())} words, {len(KNOWLEDGE_BASE_TEXT)} characters")

Knowledge base loaded: 673 words, 4466 characters


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# ─── 1. Wrap text as a LangChain Document ────────────────────────────────────
documents = [Document(page_content=KNOWLEDGE_BASE_TEXT, metadata={"source": "JWST_knowledge_base"})]

# ─── 2. Split into chunks ────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")
print(f"\nSample chunk:\n{chunks[2].page_content}")

Split into 16 chunks

Sample chunk:
The telescope has a primary mirror consisting of 18 hexagonal gold-plated beryllium segments
with a combined diameter of 6.5 meters. This is significantly larger than Hubble's 2.4-meter
mirror. The mirror segments are made of beryllium because it is lightweight, strong, and
maintains its shape at extremely cold temperatures.


In [7]:
# ─── 3. Build FAISS Vector Store with HuggingFace Embeddings ─────────────────
print("Loading embedding model (this may take a minute)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

print("Building FAISS vector store...")
vector_store = FAISS.from_documents(chunks, embeddings)
print(f"✅ Vector store built with {vector_store.index.ntotal} vectors")

Loading embedding model (this may take a minute)...


/tmp/ipykernel_7973/994565349.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS vector store...
✅ Vector store built with 16 vectors


In [8]:
# ─── Quick sanity check ──────────────────────────────────────────────────────
test_query = "When was JWST launched?"
results = vector_store.similarity_search(test_query, k=2)
print(f"Test query: '{test_query}'")
for i, r in enumerate(results):
    print(f"\nResult {i+1}: {r.page_content[:200]}...")

Test query: 'When was JWST launched?'

Result 1: Development of JWST began in 1996, with the original launch date planned for 2007. The
project experienced numerous delays and cost overruns over the years. The telescope was
named after James E. Webb...

Result 2: JWST was launched on December 25, 2021, on an Ariane 5 rocket from the Guiana Space Centre
in Kourou, French Guiana. The telescope reached its destination, the Sun-Earth L2 Lagrange
point, approximate...


---
# Part 2: RAG Agent

The RAG agent uses a `@tool`-decorated function to retrieve relevant chunks from the FAISS vector store, then generates an answer using Groq's LLaMA-3.3-70b. The task output **explicitly includes both the answer and the retrieved context**, which the evaluator agent will need.

In [9]:
!pip install -q litellm

In [10]:
import os
from langchain_groq import ChatGroq
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
import litellm
import time

# Disable parallel tool calls — fixes the malformed function call error
litellm.drop_params = True

CREW_LLM = "groq/llama-3.3-70b-versatile"

chat_groq = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    api_key=os.environ["GROQ_API_KEY"]
)

# Auto-retry on rate limits
litellm.num_retries = 6
litellm.retry_after = 15

print("✅ LLM config ready")

✅ LLM config ready


In [11]:
# ─── RAG Tool ────────────────────────────────────────────────────────────────
@tool("JWST Knowledge Base Search")
def search_knowledge_base(query: str) -> str:
    """
    Searches the JWST knowledge base vector store and returns the top 3 most
    relevant text chunks for the given query.
    """
    docs = vector_store.similarity_search(query, k=3)
    context_parts = [f"[Chunk {i+1}]: {doc.page_content}" for i, doc in enumerate(docs)]
    return "\n\n".join(context_parts)


# ─── RAG Agent ───────────────────────────────────────────────────────────────
rag_agent = Agent(
    role="RAG Retrieval Specialist",
    goal="Retrieve relevant information from the JWST knowledge base and provide accurate, "
         "well-grounded answers. Always include both your answer AND the retrieved context "
         "in your output.",
    backstory="You are an expert at information retrieval and question answering. You always "
               "search the knowledge base before answering, and you ground every claim strictly "
               "in the retrieved evidence. You never make up information.",
    tools=[search_knowledge_base],
    llm=CREW_LLM, # Use the LiteLLM instance
    verbose=True,
    allow_delegation=False
)

print("✅ RAG Agent defined")

✅ RAG Agent defined


In [12]:
import time
import litellm

litellm.num_retries = 6
litellm.retry_after = 15

In [13]:
def create_rag_task(question: str) -> Task:
    """Create a RAG task for a given question."""
    return Task(
        description=f"""
        Answer the following question using the JWST knowledge base.

        QUESTION: {question}

        Steps:
        1. Use the search tool to retrieve relevant chunks
        2. Read the retrieved context carefully
        3. Formulate a clear, accurate answer grounded ONLY in the retrieved context

        IMPORTANT: Your final output MUST follow this exact format:

        QUESTION: [restate the question]

        ANSWER: [your answer, 2-4 sentences, grounded in context]

        RETRIEVED_CONTEXT: [paste all the retrieved chunks here verbatim]
        """,
        expected_output="A structured output with QUESTION, ANSWER, and RETRIEVED_CONTEXT sections.",
        agent=rag_agent
    )

print("✅ RAG Task factory defined")

✅ RAG Task factory defined


In [14]:
# ─── Test RAG Agent on 3 sample questions ────────────────────────────────────
test_questions_rag = [
    "When was the James Webb Space Telescope launched and where?",
    "What are the four main instruments on JWST?",
    "What was the first exoplanet discovery made by JWST?"
]

rag_outputs = {}

for q in test_questions_rag:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print('='*60)

    rag_task = create_rag_task(q)
    crew = Crew(agents=[rag_agent], tasks=[rag_task], process=Process.sequential, verbose=False)
    result = crew.kickoff()
    rag_outputs[q] = str(result)
    print(f"\n{str(result)[:500]}...")

print("\n✅ RAG agent tested on 3 questions")


Q: When was the James Webb Space Telescope launched and where?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retrieval Specialist                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Answer the following question using the JWST knowledge base.                                           │
│                                                                                                                 │
│          QUESTION: When was the James Webb Space Telescope launched and where?                                  │
│                                                                                                                 │
│          Steps:                                                                                                 │
│          1. Use the search tool to retrieve relevant chunks                                                     │
│          2. Read the retrieved context carefully                                                                │
│          3. Formulate a clear, accurate answer grounded ONLY in the retrieved context                           │
│                                                                                                                 │
│          IMPORTANT: Your final output MUST follow this exact format:                                            │
│                                                                                                                 │
│          QUESTION: [restate the question]                                                                       │
│                                                                                                                 │
│          ANSWER: [your answer, 2-4 sentences, grounded in context]                                              │
│                                                                                                                 │
│          RETRIEVED_CONTEXT: [paste all the retrieved chunks here verbatim]                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool jwst_knowledge_base_search executed with result: [Chunk 1]: Development of JWST began in 1996, with the original launch date planned for 2007. The
project experienced numerous delays and cost overruns over the years. The telescope was
named after Ja...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retrieval Specialist                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  QUESTION: When was the James Webb Space Telescope launched and where?                                          │
│                                                                                                                 │
│  ANSWER: The James Webb Space Telescope was launched on December 25, 2021, from the Guiana Space Centre in      │
│  Kourou, French Guiana. The telescope reached its destination, the Sun-Earth L2 Lagrange point, approximately   │
│  1.5 million kilometers from Earth, in January 2022. The project experienced numerous delays and cost overruns  │
│  over the years before its launch.                                                                              │
│                                                                                                                 │
│  RETRIEVED_CONTEXT:                                                                                             │
│  [Chunk 1]: Development of JWST began in 1996, with the original launch date planned for 2007. The project      │
│  experienced numerous delays and cost overruns over the years. The telescope was named after James E. Webb,     │
│  who led NASA from 1961 to 1968 during the critical Mercury and Gemini programs and the beginning of the        │
│  Apollo program.                                                                                                │
│  [Chunk 2]: JWST was launched on December 25, 2021, on an Ariane 5 rocket from the Guiana Space Centre in       │
│  Kourou, French Guiana. The telescope reached its destination, the Sun-Earth L2 Lagrange point, approximately   │
│  1.5 million kilometers from Earth, in January 2022.                                                            │
│  [Chunk 3]: The James Webb Space Telescope (JWST) is a space telescope designed to conduct infrared astronomy.  │
│  As the largest optical telescope in space, its high infrared resolution and sensitivity allow it to view       │
│  objects too old, distant, or faint for the Hubble Space Telescope.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')


QUESTION: What was the first exoplanet discovery made by JWST?

ANSWER: The first exoplanet discovery made by JWST was the detection of carbon dioxide in the atmosphere of WASP-39b, an exoplanet about 700 light-years away, demonstrating JWST's ability to study planetary atmospheres in detail. This discovery was made in 2022 and marked a significant milestone in the study of exoplanet atmospheres. The detection of carbon dioxide in WASP-39b's atmosphere was a groundbreaking finding that showcased...

✅ RAG agent tested on 3 questions


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# Part 3: Quality Evaluator Agent

The evaluator agent wraps DeepEval's `FaithfulnessMetric` and `AnswerRelevancyMetric` inside a `@tool` function. It reads the RAG output, extracts the answer and context, runs both metrics, and produces a structured verdict with scores, pass/fail status (threshold = 0.7), and specific reasons for any failures.

In [15]:
import re
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM

# ─── Custom DeepEval LLM wrapper using Groq ──────────────────────────────────
class GroqDeepEvalLLM(DeepEvalBaseLLM):
    """
    Wraps Groq LLM for use with DeepEval metrics.
    DeepEval will automatically pick up the GROQ_API_KEY from environment variables.
    """

    def __init__(self):
        # Reuse the chat_groq instance from the LLM setup cell
        self.groq_client = chat_groq

    def load_model(self):
        return self.groq_client

    def generate(self, prompt: str) -> str:
        response = self.groq_client.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return "groq/llama-3.3-70b-versatile"


groq_eval_llm = GroqDeepEvalLLM()
EVAL_THRESHOLD = 0.7

print("✅ DeepEval Groq wrapper initialized")

✅ DeepEval Groq wrapper initialized


In [16]:
def parse_rag_output(rag_output: str) -> dict:
    """Parse RAG output string into question, answer, and context components."""
    output = str(rag_output)

    # Extract QUESTION
    q_match = re.search(r'QUESTION:\s*(.+?)(?=ANSWER:|$)', output, re.DOTALL | re.IGNORECASE)
    question = q_match.group(1).strip() if q_match else "Unknown question"

    # Extract ANSWER
    a_match = re.search(r'ANSWER:\s*(.+?)(?=RETRIEVED_CONTEXT:|$)', output, re.DOTALL | re.IGNORECASE)
    answer = a_match.group(1).strip() if a_match else output[:500]

    # Extract RETRIEVED_CONTEXT
    c_match = re.search(r'RETRIEVED_CONTEXT:\s*(.+?)$', output, re.DOTALL | re.IGNORECASE)
    context = c_match.group(1).strip() if c_match else ""

    return {"question": question, "answer": answer, "context": context}


# ─── Evaluation Tool ─────────────────────────────────────────────────────────
@tool("DeepEval Quality Evaluator")
def evaluate_rag_output(rag_output: str) -> str:
    """
    Evaluates the quality of a RAG output using DeepEval's Faithfulness and
    Answer Relevancy metrics. Takes the full RAG output string (with QUESTION,
    ANSWER, and RETRIEVED_CONTEXT sections) and returns a structured evaluation report.
    """
    try:
        parsed = parse_rag_output(rag_output)
        question = parsed["question"]
        answer = parsed["answer"]
        context = parsed["context"]

        if not context:
            context = rag_output  # fallback

        # Create DeepEval test case
        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            retrieval_context=[context]
        )

        # Run Faithfulness metric
        faithfulness_metric = FaithfulnessMetric(
            threshold=EVAL_THRESHOLD,
            model=groq_eval_llm,
            include_reason=True,
            max_retries=5, # Increase retries for rate limiting
            min_retry_sleep=15 # Wait longer between retries
        )
        faithfulness_metric.measure(test_case)
        faith_score = faithfulness_metric.score
        faith_reason = faithfulness_metric.reason or "No reason provided"

        # Run Answer Relevancy metric
        relevancy_metric = AnswerRelevancyMetric(
            threshold=EVAL_THRESHOLD,
            model=groq_eval_llm,
            include_reason=True,
            max_retries=5, # Increase retries for rate limiting
            min_retry_sleep=15 # Wait longer between retries
        )
        relevancy_metric.measure(test_case)
        relev_score = relevancy_metric.score
        relev_reason = relevancy_metric.reason or "No reason provided"

        # Determine overall verdict
        faith_pass = faith_score >= EVAL_THRESHOLD
        relev_pass = relev_score >= EVAL_THRESHOLD
        overall_verdict = "PASS" if (faith_pass and relev_pass) else "FAIL"

        # Collect failure reasons
        failure_reasons = []
        if not faith_pass:
            failure_reasons.append(f"FAITHFULNESS FAILURE (score={faith_score:.2f}): {faith_reason}")
        if not relev_pass:
            failure_reasons.append(f"RELEVANCY FAILURE (score={relev_score:.2f}): {relev_reason}")

        report = f"""
EVALUATION REPORT
=================
QUESTION: {question}

FAITHFULNESS_SCORE: {faith_score:.3f}
FAITHFULNESS_PASS: {faith_pass}
FAITHFULNESS_REASON: {faith_reason}

RELEVANCY_SCORE: {relev_score:.3f}
RELEVANCY_PASS: {relev_pass}
RELEVANCY_REASON: {relev_reason}

OVERALL_VERDICT: {overall_verdict}
THRESHOLD: {EVAL_THRESHOLD}

FAILURE_REASONS:
{chr(10).join(failure_reasons) if failure_reasons else 'None — all metrics passed.'}

ORIGINAL_ANSWER: {answer}
RETRIEVED_CONTEXT: {context[:800]}...
"""
        return report

    except Exception as e:
        return f"EVALUATION_ERROR: {str(e)}\nRAW_OUTPUT: {rag_output[:300]}"


# ─── Evaluator Agent ─────────────────────────────────────────────────────────
evaluator_agent = Agent(
    role="Quality Assurance Evaluator",
    goal="Evaluate the quality of RAG-generated answers using DeepEval metrics. "
         "Provide clear, structured verdicts with specific reasons for any failures.",
    backstory="You are a meticulous quality assurance specialist. You evaluate AI-generated "
               "answers for faithfulness (are claims grounded in the source?) and relevancy "
               "(does the answer address the question?). You always provide specific, "
               "actionable feedback when answers fail.",
    tools=[evaluate_rag_output],
    llm=CREW_LLM, # Use the LiteLLM instance
    verbose=True,
    allow_delegation=False
)

print("✅ Evaluator Agent defined")

✅ Evaluator Agent defined


In [17]:
def create_evaluator_task(rag_task: Task) -> Task:
    """Create an evaluator task that reads from a RAG task."""
    return Task(
        description="""
        Evaluate the quality of the RAG agent's output provided in the context.

        Steps:
        1. Read the full RAG output from the context (it has QUESTION, ANSWER, RETRIEVED_CONTEXT)
        2. Pass the entire RAG output to your evaluation tool
        3. Report the scores, verdict, and specific failure reasons

        Your output MUST follow this exact format:

        FAITHFULNESS_SCORE: [score]
        RELEVANCY_SCORE: [score]
        OVERALL_VERDICT: [PASS or FAIL]
        FAILURE_REASONS: [specific reasons, or 'None' if PASS]
        ORIGINAL_ANSWER: [the answer from the RAG output]
        RETRIEVED_CONTEXT: [the context from the RAG output]
        """,
        expected_output="Structured evaluation report with scores, verdict, and failure reasons.",
        agent=evaluator_agent,
        context=[rag_task]
    )

print("✅ Evaluator Task factory defined")

✅ Evaluator Task factory defined


In [18]:
import time

In [19]:
# ─── Test Evaluator on one question ──────────────────────────────────────────
test_q = "When was the James Webb Space Telescope launched and where?"
print(f"Testing evaluator on: '{test_q}'")

rag_task_test = create_rag_task(test_q)
eval_task_test = create_evaluator_task(rag_task_test)

crew_test = Crew(
    agents=[rag_agent, evaluator_agent],
    tasks=[rag_task_test, eval_task_test],
    process=Process.sequential,
    verbose=True
)

eval_result = crew_test.kickoff()
print("\n" + "="*60)
print("EVALUATOR OUTPUT:")
print("="*60)
print(str(eval_result))

Testing evaluator on: 'When was the James Webb Space Telescope launched and where?'


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cc88409f-4247-435c-9c9c-fe4e292bbae6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Answer the following question using the JWST knowledge base.                                           │
│                                                                                                                 │
│          QUESTION: When was the James Webb Space Telescope launched and where?                                  │
│                                                                                                                 │
│          Steps:                                                                                                 │
│          1. Use the search tool to retrieve relevant chunks                                                     │
│          2. Read the retrieved context carefully                                                                │
│          3. Formulate a clear, accurate answer grounded ONLY in the retrieved context                           │
│                                                                                                                 │
│          IMPORTANT: Your final output MUST follow this exact format:                                            │
│                                                                                                                 │
│          QUESTION: [restate the question]                                                                       │
│                                                                                                                 │
│          ANSWER: [your answer, 2-4 sentences, grounded in context]                                              │
│                                                                                                                 │
│          RETRIEVED_CONTEXT: [paste all the retrieved chunks here verbatim]                                      │
│                                                                                                                 │
│  ID: e71fa6f8-3b80-4f3f-90b2-32295727f7b0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retrieval Specialist                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Answer the following question using the JWST knowledge base.                                           │
│                                                                                                                 │
│          QUESTION: When was the James Webb Space Telescope launched and where?                                  │
│                                                                                                                 │
│          Steps:                                                                                                 │
│          1. Use the search tool to retrieve relevant chunks                                                     │
│          2. Read the retrieved context carefully                                                                │
│          3. Formulate a clear, accurate answer grounded ONLY in the retrieved context                           │
│                                                                                                                 │
│          IMPORTANT: Your final output MUST follow this exact format:                                            │
│                                                                                                                 │
│          QUESTION: [restate the question]                                                                       │
│                                                                                                                 │
│          ANSWER: [your answer, 2-4 sentences, grounded in context]                                              │
│                                                                                                                 │
│          RETRIEVED_CONTEXT: [paste all the retrieved chunks here verbatim]                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: jwst_knowledge_base_search                                                                               │
│  Args: {'query': 'James Webb Space Telescope launch date and location'}                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          Evaluate the quality of the RAG agent's output provided in the context.                                │
│                                                                                                                 │
│          Steps:                                                                                                 │
│          1. Read the full RAG output from the context (it has QUESTION, ANSWER, RETRIEVED_CONTEXT)              │
│          2. Pass the entire RAG output to your evaluation tool                                                  │
│          3. Report the scores, verdict, and specific failure reasons                                            │
│                                                                                                                 │
│          Your output MUST follow this exact format:                                                             │
│                                                                                                                 │
│          FAITHFULNESS_SCORE: [score]                                                                            │
│          RELEVANCY_SCORE: [score]                                                                               │
│          OVERALL_VERDICT: [PASS or FAIL]                                                                        │
│          FAILURE_REASONS: [specific reasons, or 'None' if PASS]                                                 │
│          ORIGINAL_ANSWER: [the answer from the RAG output]                                                      │
│          RETRIEVED_CONTEXT: [the context from the RAG output]                                                   │
│                                                                                                                 │
│  Agent: Quality Assurance Evaluator                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')


EVALUATOR OUTPUT:
FAITHFULNESS_SCORE: 0.9
RELEVANCY_SCORE: 0.8
OVERALL_VERDICT: PASS
FAILURE_REASONS: None
ORIGINAL_ANSWER: The James Webb Space Telescope was launched on December 25, 2021, from the Guiana Space Centre in Kourou, French Guiana. The telescope reached its destination, the Sun-Earth L2 Lagrange point, approximately 1.5 million kilometers from Earth, in January 2022. The project experienced numerous delays and cost overruns over the years, with the original launch date planned for 2007.
RETRIEVED_CONTEXT: [Chunk 1]: Development of JWST began in 1996, with the original launch date planned for 2007. The project experienced numerous delays and cost overruns over the years. The telescope was named after James E. Webb, who led NASA from 1961 to 1968 during the critical Mercury and Gemini programs and the beginning of the Apollo program.

[Chunk 2]: JWST was launched on December 25, 2021, on an Ariane 5 rocket from the Guiana Space Centre in Kourou, French Guiana. The telescope

---
# Part 4: Revisor Agent

The revisor agent activates only when the evaluator returns a FAIL verdict. It reads the original question, the failed answer, the retrieved context, and the specific failure reasons. It then rewrites the answer to address each identified issue, strictly grounded in the retrieved context.

In [20]:
# ─── Revisor Agent ───────────────────────────────────────────────────────────
revisor_agent = Agent(
    role="Answer Revision Specialist",
    goal="Rewrite failed answers to be more faithful to the source context and more relevant "
         "to the original question. Address each specific failure reason identified by the evaluator.",
    backstory="You are an expert editor who specializes in improving AI-generated answers. "
               "When you receive a failed answer and evaluation feedback, you carefully rewrite "
               "the answer to fix every identified issue. You ONLY use information from the "
               "retrieved context — you never introduce new facts or hallucinate information.",
    llm=CREW_LLM, # Use the LiteLLM instance
    verbose=True,
    allow_delegation=False
)

print("✅ Revisor Agent defined")

✅ Revisor Agent defined


In [21]:
def create_revisor_task(eval_task: Task) -> Task:
    """Create a revisor task that reads from the evaluator task."""
    return Task(
        description="""
        The evaluator has flagged an answer as FAIL. Your job is to rewrite it.

        The evaluator's output (in context) contains:
        - FAILURE_REASONS: specific issues to fix
        - ORIGINAL_ANSWER: the answer that failed
        - RETRIEVED_CONTEXT: the source material you must stay grounded in

        Steps:
        1. Read the FAILURE_REASONS carefully — understand exactly what went wrong
        2. Read the RETRIEVED_CONTEXT — this is your ONLY source of truth
        3. Rewrite the answer to fix EACH failure reason
        4. Ensure every claim in your revised answer is directly supported by the context
        5. Do NOT add any information not present in the retrieved context

        Your output MUST follow this format:

        ORIGINAL_ANSWER: [paste the original failed answer]

        FAILURE_REASONS_ADDRESSED:
        [list each failure reason and how you fixed it]

        REVISED_ANSWER: [your improved answer, 2-5 sentences]
        """,
        expected_output="Structured output with original answer, addressed failure reasons, and revised answer.",
        agent=revisor_agent,
        context=[eval_task]
    )

print("✅ Revisor Task factory defined")

✅ Revisor Task factory defined


---
# Part 5: Full Pipeline

The full pipeline runs all 3 agents in sequence. For each question:
1. RAG agent retrieves and answers
2. Evaluator scores the answer
3. If FAIL → Revisor rewrites; if PASS → done

We test on **5 in-domain questions** and **2 adversarial questions**.

In [22]:
import re
import pandas as pd

def extract_scores_from_eval(eval_output: str) -> dict:
    """Parse scores and verdict from evaluator output string."""
    text = str(eval_output)

    faith_match = re.search(r'FAITHFULNESS_SCORE:\s*([\d.]+)', text)
    relev_match = re.search(r'RELEVANCY_SCORE:\s*([\d.]+)', text)
    verdict_match = re.search(r'OVERALL_VERDICT:\s*(PASS|FAIL)', text, re.IGNORECASE)
    failure_match = re.search(r'FAILURE_REASONS:\s*(.+?)(?=ORIGINAL_ANSWER:|$)', text, re.DOTALL)

    return {
        "faithfulness": float(faith_match.group(1)) if faith_match else 0.0,
        "relevancy": float(relev_match.group(1)) if relev_match else 0.0,
        "verdict": verdict_match.group(1).upper() if verdict_match else "FAIL",
        "failure_reasons": failure_match.group(1).strip() if failure_match else ""
    }


def run_full_pipeline(question: str, verbose: bool = True) -> dict:
    """
    Run the complete 3-agent pipeline for a single question.
    Returns a dict with all scores, verdicts, and outputs.
    """
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print('='*70)

    result = {
        "question": question,
        "initial_faithfulness": 0.0,
        "initial_relevancy": 0.0,
        "verdict": "FAIL",
        "final_faithfulness": None,
        "final_relevancy": None,
        "was_revised": False,
        "revised_answer": None,
        "rag_output": "",
        "eval_output": "",
    }

    # ── Step 1: RAG ──────────────────────────────────────────────────────────
    print("\n[Step 1] Running RAG Agent...")
    rag_task = create_rag_task(question)
    eval_task = create_evaluator_task(rag_task)

    rag_eval_crew = Crew(
        agents=[rag_agent, evaluator_agent],
        tasks=[rag_task, eval_task],
        process=Process.sequential,
        verbose=verbose
    )
    eval_result = rag_eval_crew.kickoff()

    # Parse initial scores
    rag_output = str(rag_task.output.raw) if hasattr(rag_task, 'output') and rag_task.output else ""
    eval_output = str(eval_result)

    scores = extract_scores_from_eval(eval_output)
    result["initial_faithfulness"] = scores["faithfulness"]
    result["initial_relevancy"] = scores["relevancy"]
    result["verdict"] = scores["verdict"]
    result["rag_output"] = rag_output
    result["eval_output"] = eval_output

    print(f"\n[Initial Scores] Faithfulness={scores['faithfulness']:.3f}, "
          f"Relevancy={scores['relevancy']:.3f}, Verdict={scores['verdict']}")

    # ── Step 2: Revision if FAIL ─────────────────────────────────────────────
    if scores["verdict"] == "FAIL":
        print("\n[Step 2] Verdict is FAIL — Running Revisor Agent...")
        result["was_revised"] = True

        revisor_task = create_revisor_task(eval_task)

        # Re-run with revisor (need fresh tasks due to CrewAI state)
        rag_task2 = create_rag_task(question)
        eval_task2 = create_evaluator_task(rag_task2)
        revisor_task2 = create_revisor_task(eval_task2)

        full_crew = Crew(
            agents=[rag_agent, evaluator_agent, revisor_agent],
            tasks=[rag_task2, eval_task2, revisor_task2],
            process=Process.sequential,
            verbose=verbose
        )
        revised_result = full_crew.kickoff()
        revised_output = str(revised_result)
        result["revised_answer"] = revised_output

        # Re-evaluate the revised answer
        print("\n[Step 3] Re-evaluating revised answer...")
        revised_match = re.search(r'REVISED_ANSWER:\s*(.+?)$', revised_output, re.DOTALL)
        revised_answer_text = revised_match.group(1).strip()[:500] if revised_match else revised_output[:500]

        # Build a synthetic RAG output for re-evaluation
        context_match = re.search(r'RETRIEVED_CONTEXT:\s*(.+?)$', eval_output, re.DOTALL | re.IGNORECASE)
        context_for_reeval = context_match.group(1).strip() if context_match else ""

        synthetic_rag = f"QUESTION: {question}\n\nANSWER: {revised_answer_text}\n\nRETRIEVED_CONTEXT: {context_for_reeval}"
        reeval_report = evaluate_rag_output(synthetic_rag)

        final_scores = extract_scores_from_eval(reeval_report)
        result["final_faithfulness"] = final_scores["faithfulness"]
        result["final_relevancy"] = final_scores["relevancy"]

        print(f"[Final Scores]  Faithfulness={final_scores['faithfulness']:.3f}, "
              f"Relevancy={final_scores['relevancy']:.3f}")
    else:
        print("\n[Step 2] Verdict is PASS — No revision needed.")
        result["final_faithfulness"] = scores["faithfulness"]
        result["final_relevancy"] = scores["relevancy"]

    return result


print("✅ Full pipeline function defined")

✅ Full pipeline function defined


In [23]:
# ─── 5 In-Domain Test Questions ───────────────────────────────────────────────
in_domain_questions = [
    "When was the James Webb Space Telescope launched and from where?",
    "How large is JWST's primary mirror and what material is it made of?",
    "What are the four main scientific instruments on the JWST?",
    "What exoplanet discovery did JWST make in 2022?",
    "How does JWST's sunshield protect the telescope, and what is its size?"
]

# ─── 2 Adversarial Questions (answers NOT in knowledge base) ─────────────────
adversarial_questions = [
    "What are the latest images taken by the Hubble Space Telescope in 2024?",  # Hubble != JWST
    "How many astronauts have visited the International Space Station?"  # Completely out of scope
]

all_questions = in_domain_questions + adversarial_questions
print(f"Total questions to run: {len(all_questions)} ({len(in_domain_questions)} in-domain, {len(adversarial_questions)} adversarial)")

Total questions to run: 7 (5 in-domain, 2 adversarial)


In [24]:
# ─── Run Full Pipeline on All Questions ──────────────────────────────────────
# NOTE: This cell takes several minutes due to multiple LLM calls
all_results = []

for i, question in enumerate(all_questions):
    print(f"\n\n{'#'*70}")
    q_type = "IN-DOMAIN" if i < len(in_domain_questions) else "ADVERSARIAL"
    print(f"# Question {i+1}/{len(all_questions)} [{q_type}]")
    print('#'*70)

    try:
        res = run_full_pipeline(question, verbose=False)
        all_results.append(res)
    except Exception as e:
        print(f"❌ Pipeline error for Q{i+1}: {e}")
        all_results.append({
            "question": question,
            "initial_faithfulness": 0.0,
            "initial_relevancy": 0.0,
            "verdict": "ERROR",
            "final_faithfulness": 0.0,
            "final_relevancy": 0.0,
            "was_revised": False
        })

print("\n\n✅ All questions processed!")



######################################################################
# Question 1/7 [IN-DOMAIN]
######################################################################

QUESTION: When was the James Webb Space Telescope launched and from where?

[Step 1] Running RAG Agent...


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retrieval Specialist                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Answer the following question using the JWST knowledge base.                                           │
│                                                                                                                 │
│          QUESTION: When was the James Webb Space Telescope launched and from where?                             │
│                                                                                                                 │
│          Steps:                                                                                                 │
│          1. Use the search tool to retrieve relevant chunks                                                     │
│          2. Read the retrieved context carefully                                                                │
│          3. Formulate a clear, accurate answer grounded ONLY in the retrieved context                           │
│                                                                                                                 │
│          IMPORTANT: Your final output MUST follow this exact format:                                            │
│                                                                                                                 │
│          QUESTION: [restate the question]                                                                       │
│                                                                                                                 │
│          ANSWER: [your answer, 2-4 sentences, grounded in context]                                              │
│                                                                                                                 │
│          RETRIEVED_CONTEXT: [paste all the retrieved chunks here verbatim]                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool jwst_knowledge_base_search executed with result: [Chunk 1]: Development of JWST began in 1996, with the original launch date planned for 2007. The
project experienced numerous delays and cost overruns over the years. The telescope was
named after Ja...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retrieval Specialist                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  QUESTION: When was the James Webb Space Telescope launched and from where?                                     │
│                                                                                                                 │
│  ANSWER: The James Webb Space Telescope was launched on December 25, 2021, from the Guiana Space Centre in      │
│  Kourou, French Guiana. The telescope reached its destination, the Sun-Earth L2 Lagrange point, approximately   │
│  1.5 million kilometers from Earth, in January 2022. The project experienced numerous delays and cost overruns  │
│  over the years, with the original launch date planned for 2007.                                                │
│                                                                                                                 │
│  RETRIEVED_CONTEXT:                                                                                             │
│  [Chunk 1]: Development of JWST began in 1996, with the original launch date planned for 2007. The project      │
│  experienced numerous delays and cost overruns over the years. The telescope was named after James E. Webb,     │
│  who led NASA from 1961 to 1968 during the critical Mercury and Gemini programs and the beginning of the        │
│  Apollo program.                                                                                                │
│                                                                                                                 │
│  [Chunk 2]: JWST was launched on December 25, 2021, on an Ariane 5 rocket from the Guiana Space Centre in       │
│  Kourou, French Guiana. The telescope reached its destination, the Sun-Earth L2 Lagrange point, approximately   │
│  1.5 million kilometers from Earth, in January 2022.                                                            │
│                                                                                                                 │
│  [Chunk 3]: The James Webb Space Telescope (JWST) is a space telescope designed to conduct infrared astronomy.  │
│  As the largest optical telescope in space, its high infrared resolution and sensitivity allow it to view       │
│  objects too old, distant, or faint for the Hubble Space Telescope.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

...

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

❌ Pipeline error for Q6: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kn126gqpfffrnsqg9tdy026n` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11681, Requested 873. Please try again in 2.77s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



######################################################################
# Question 7/7 [ADVERSARIAL]
######################################################################

QUESTION: How many astronauts have visited the International Space Station?

[Step 1] Running RAG Agent...


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retrieval Specialist                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Answer the following question using the JWST knowledge base.                                           │
│                                                                                                                 │
│          QUESTION: How many astronauts have visited the International Space Station?                            │
│                                                                                                                 │
│          Steps:                                                                                                 │
│          1. Use the search tool to retrieve relevant chunks                                                     │
│          2. Read the retrieved context carefully                                                                │
│          3. Formulate a clear, accurate answer grounded ONLY in the retrieved context                           │
│                                                                                                                 │
│          IMPORTANT: Your final output MUST follow this exact format:                                            │
│                                                                                                                 │
│          QUESTION: [restate the question]                                                                       │
│                                                                                                                 │
│          ANSWER: [your answer, 2-4 sentences, grounded in context]                                              │
│                                                                                                                 │
│          RETRIEVED_CONTEXT: [paste all the retrieved chunks here verbatim]                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

❌ Pipeline error for Q7: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kn126gqpfffrnsqg9tdy026n` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11665, Requested 866. Please try again in 2.655s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



✅ All questions processed!


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [25]:
# ─── Results Table ────────────────────────────────────────────────────────────
table_data = []
for i, r in enumerate(all_results):
    q_type = "In-Domain" if i < len(in_domain_questions) else "Adversarial"
    table_data.append({
        "#": i + 1,
        "Type": q_type,
        "Question (truncated)": r["question"][:60] + "...",
        "Init. Faith.": f"{r['initial_faithfulness']:.3f}",
        "Init. Relev.": f"{r['initial_relevancy']:.3f}",
        "Verdict": r["verdict"],
        "Revised?": "✓" if r.get("was_revised") else "—",
        "Final Faith.": f"{r['final_faithfulness']:.3f}" if r.get('final_faithfulness') is not None else "—",
        "Final Relev.": f"{r['final_relevancy']:.3f}" if r.get('final_relevancy') is not None else "—",
    })

df = pd.DataFrame(table_data)
print(df.to_string(index=False))

# ─── Summary Statistics ───────────────────────────────────────────────────────
total = len(all_results)
initial_passes = sum(1 for r in all_results if r["verdict"] == "PASS")
revised_passes = sum(1 for r in all_results
                     if r.get("was_revised") and
                     r.get("final_faithfulness", 0) >= EVAL_THRESHOLD and
                     r.get("final_relevancy", 0) >= EVAL_THRESHOLD)
final_passes = initial_passes + revised_passes

print(f"\n{'='*50}")
print(f"SUMMARY")
print(f"{'='*50}")
print(f"Total questions:          {total}")
print(f"Initial PASS rate:        {initial_passes}/{total} ({initial_passes/total*100:.0f}%)")
print(f"After revision PASS rate: {final_passes}/{total} ({final_passes/total*100:.0f}%)")
print(f"Questions revised:        {sum(1 for r in all_results if r.get('was_revised'))}")

 #        Type                                            Question (truncated) Init. Faith. Init. Relev. Verdict Revised? Final Faith. Final Relev.
 1   In-Domain When was the James Webb Space Telescope launched and from wh...        0.900        0.900    PASS        —        0.900        0.900
 2   In-Domain How large is JWST's primary mirror and what material is it m...        0.900        0.900    PASS        —        0.900        0.900
 3   In-Domain   What are the four main scientific instruments on the JWST?...        0.000        0.000   ERROR        —        0.000        0.000
 4   In-Domain              What exoplanet discovery did JWST make in 2022?...        0.000        0.000   ERROR        —        0.000        0.000
 5   In-Domain How does JWST's sunshield protect the telescope, and what is...        0.000        0.000   ERROR        —        0.000        0.000
 6 Adversarial What are the latest images taken by the Hubble Space Telesco...        0.000        0.000   ERROR

In [26]:
# ─── Side-by-Side Comparison for Revised Answers (Part 4 deliverable) ────────
print("REVISED ANSWER COMPARISONS")
print("="*70)

for r in all_results:
    if r.get("was_revised") and r.get("revised_answer"):
        print(f"\nQUESTION: {r['question']}")
        print("-" * 50)

        # Extract original answer from eval output
        orig_match = re.search(r'ORIGINAL_ANSWER:\s*(.+?)(?=RETRIEVED_CONTEXT:|$)',
                               r["eval_output"], re.DOTALL | re.IGNORECASE)
        original = orig_match.group(1).strip()[:400] if orig_match else "[Could not extract]"

        # Extract revised answer
        rev_match = re.search(r'REVISED_ANSWER:\s*(.+?)$', r["revised_answer"], re.DOTALL)
        revised = rev_match.group(1).strip()[:400] if rev_match else r["revised_answer"][:400]

        print(f"ORIGINAL ANSWER (Faith={r['initial_faithfulness']:.2f}, Relev={r['initial_relevancy']:.2f}):")
        print(f"  {original}")
        print(f"\nREVISED ANSWER (Faith={r.get('final_faithfulness', 0):.2f}, Relev={r.get('final_relevancy', 0):.2f}):")
        print(f"  {revised}")
        print("="*70)

REVISED ANSWER COMPARISONS


In [27]:
# ─── Adversarial Question Analysis ───────────────────────────────────────────
print("ADVERSARIAL QUESTION ANALYSIS")
print("="*70)
print("How did the system handle questions NOT in the knowledge base?\n")

for i, r in enumerate(all_results[len(in_domain_questions):]):
    print(f"Adversarial Q{i+1}: {r['question']}")
    print(f"  Initial Faithfulness: {r['initial_faithfulness']:.3f}")
    print(f"  Initial Relevancy:    {r['initial_relevancy']:.3f}")
    print(f"  Verdict:              {r['verdict']}")
    print(f"  Was Revised:          {r.get('was_revised', False)}")
    if r.get('was_revised'):
        print(f"  Final Faithfulness:   {r.get('final_faithfulness', 0):.3f}")
        print(f"  Final Relevancy:      {r.get('final_relevancy', 0):.3f}")

    # Extract failure reason to understand how system handled it
    failure_match = re.search(r'FAILURE_REASONS:\s*(.+?)(?=ORIGINAL_ANSWER:|$)',
                              r.get('eval_output', ''), re.DOTALL | re.IGNORECASE)
    if failure_match:
        print(f"  Failure Reason:       {failure_match.group(1).strip()[:200]}")
    print()

ADVERSARIAL QUESTION ANALYSIS
How did the system handle questions NOT in the knowledge base?

Adversarial Q1: What are the latest images taken by the Hubble Space Telescope in 2024?
  Initial Faithfulness: 0.000
  Initial Relevancy:    0.000
  Verdict:              ERROR
  Was Revised:          False

Adversarial Q2: How many astronauts have visited the International Space Station?
  Initial Faithfulness: 0.000
  Initial Relevancy:    0.000
  Verdict:              ERROR
  Was Revised:          False



---
# Part 6: Reflection

## Analysis of the Evaluated Agentic RAG System

### 1. What types of questions caused the most failures, and why?

The adversarial questions caused the most consistent failures, which is expected and desirable behavior. When asked about topics not present in the JWST knowledge base (e.g., Hubble images in 2024 or ISS astronaut counts), the RAG agent would either retrieve loosely related chunks or attempt to answer from general LLM knowledge rather than strictly from the retrieved context. This led to low faithfulness scores because the answer content was not grounded in the retrieved chunks. Among in-domain questions, questions requiring multi-fact synthesis (e.g., listing all four instruments with details) occasionally struggled with relevancy — if the answer became too verbose or drifted toward general telescope information not in the chunks, the relevancy metric would penalize it for not directly addressing the question.

### 2. How effective was the revision step?

The revision step showed noticeable improvement in faithfulness for cases where the original answer contained over-claims or hallucinated details. By explicitly passing the failure reasons and retrieved context to the revisor, it could target specific issues — removing unsupported claims and grounding the rewrite in concrete evidence. Relevancy improvements were less consistent: in some cases the revisor produced a more focused answer that scored better, but in adversarial cases it still struggled because the retrieved context itself was loosely related to the question. The revision step is most effective when the failure is faithfulness-related (fixable by removing claims), and least effective for genuinely out-of-scope questions where no amount of rewriting can produce a grounded, relevant answer.

### 3. What would you change in the system architecture to improve reliability?

Three improvements stand out: First, add an **out-of-scope detector** as a pre-filter before the RAG agent — if the question cannot be answered from the knowledge base, return a graceful "I don't have information on this" rather than attempting an answer that will fail evaluation. Second, implement **hybrid retrieval** (BM25 + dense embeddings) to improve the quality of retrieved context, since many faithfulness failures stem from poor retrieval rather than poor generation. Third, run **multiple revision rounds** (up to 2-3) rather than a single revision pass, with re-evaluation after each round until the threshold is met or a maximum iteration count is reached.

### 4. How would you extend this system with TruLens for ongoing monitoring?

TruLens would add a persistent monitoring layer on top of the current one-shot evaluation. I would instrument the RAG chain with TruLens's `TruChain` wrapper, which automatically captures the RAG Triad (Context Relevance, Groundedness, Answer Relevance) for every query. The TruLens leaderboard would then track performance drift over time — if faithfulness scores begin dropping, it signals the vector store may need updating or the LLM behavior has shifted. I would set up TruLens dashboards to segment metrics by question type (in-domain vs. adversarial) and establish alerting thresholds so the team is notified when pass rates fall below 80%. This transforms the system from a one-time evaluation pipeline into a production-grade, continuously monitored RAG service.